In [20]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.tools import TavilySearchResults, StructuredTool
from langchain.memory import ConversationBufferMemory
from pydantic import BaseModel

In [37]:
#Load API key
load_dotenv(dotenv_path=".env")
openai_api_key=os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

#Initialize LLM
llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [33]:
#Simple QA Tool

qa_prompt=PromptTemplate.from_template("Answer clearly: {question}")
qa_chain=LLMChain(llm=llm, prompt=qa_prompt)
qa_tool=Tool(
    name="Simple_QA",
    func=qa_chain.run,
    description="Answer factual questions clearly"
)


#Web Search (Tavily)
search_tool=Tool(
    name="Web_Search",
    func=TavilySearchResults(max_results=3).run,
    description="Search the internet for current information"
)

#Wrap all tools in an agent
tools=[qa_tool,search_tool]

#Memory
memory= ConversationBufferMemory(memory_key="chat_history", return_message=True, output_key="output")


In [8]:
#1) Zero-Shot ReAct (simple + reliable default)

agent=initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

result=agent.run("Summarize LangChain in 2 lines, then tell me who created it.")

C:\Users\mdfai\AppData\Local\Temp\ipykernel_11048\1088419892.py:10: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result=agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...
I should search the internet for information on LangChain and its creator.
Action: Web Search
Action Input: "LangChain summary" and "LangChain creator"
Observation: [{'title': 'LangChain overview - Docs by LangChain', 'url': 'https://docs.langchain.com/oss/python/langchain/overview', 'content': 'Copy page\n\nAgent = Model + Harness. LangChain provides `create_agent`: a minimal, highly configurable harness. The harness is everything around the model loop: the prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what your use case needs. Supports OpenAI, Anthropic, Google, and more. [...] Overview\n   Prebuilt middleware\n   Custom middleware\n\n### Frontend\n\n   Overview\n   Patterns  \n   Integrations  \n\n### Advanced usage\n\n   Guardrails\n   Runtime\n   Context engineering\n   Model Context Protocol (MCP)\n   Human-in-the-loop\n   Multi-agent  \n   Retrieval\n   Long-term memory\n\n### Age

In [9]:
#2) Conversational ReAct (with memory; non-chat model baseline)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: LangChain is an agent engineering platform created by Harrison Chase, who is the CEO and co-founder of LangChain.

> Finished chain.


In [10]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)

result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")



> Entering new AgentExecutor chain...
```json
{
    "action": "Simple QA",
    "action_input": "What is LangChain and who created it?"
}
```
Observation: LangChain is an open-source framework designed for building applications that utilize large language models (LLMs). It provides tools and components to facilitate the integration of LLMs with various data sources, APIs, and workflows, enabling developers to create more complex and interactive applications.

LangChain was created by Harrison Chase and is maintained by a community of contributors.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "LangChain is an open-source framework designed for building applications that utilize large language models (LLMs). It was created by Harrison Chase and is maintained by a community of contributors."
}
```

> Finished chain.


In [16]:
#4.Structured Chat Zero Shot

#Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str= "concise"

#Define the tool function
def title_tool_fn(topic: str, tone: str= "concise") ->  str:
    return f"{tone.title()} Title:{topic} in practice"

# Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)

# Initialize the agent

agent= initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)



> Entering new AgentExecutor chain...
Thought: I can use the TitleMaker tool to generate a friendly title about LangGraph tutorials.

Action:
```
{
  "action": "TitleMaker",
  "action_input": {"topic": "LangGraph tutorials", "tone": "friendly"}
}
```


Observation: Friendly Title:LangGraph tutorials in practice
Thought:I can provide the final response to the user's request for a friendly title about LangGraph tutorials.
Action:
```
{
  "action": "Final Answer",
  "action_input": "Friendly Title: LangGraph tutorials in practice"
}
```

> Finished chain.
✅ Result: Friendly Title: LangGraph tutorials in practice


In [34]:
#5)OpenAI Functions (function-calling schema)

agent = initialize_agent(
    tools= tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)
result = agent.run("Search the web for LangGraph docs and give me 3 bullets.")




> Entering new AgentExecutor chain...

Invoking: `Web_Search` with `LangGraph documentation`


[{'title': 'LangGraph overview - Docs by LangChain', 'url': 'https://docs.langchain.com/oss/python/langgraph/overview', 'content': 'Trusted by companies shaping the future of agents— including Klarna, Uber, J.P. Morgan, and more— LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents. LangGraph is very low-level, and focused entirely on agent orchestration. Before using LangGraph, we recommend you familiarize yourself with some of the components used to build agents, starting with models and tools. We will commonly use LangChain components throughout the documentation to integrate models and tools, but you don’t need to use LangChain to use LangGraph. If you are just getting started with agents or want a higher-level abstraction, we recommend you use LangChain’s agents that provide prebuilt architectures for common LL

In [36]:
#6) OpenAI Multi-Functions (can call multiple tools in one step)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,
    verbose=True
)
result = agent.run("captial of Netherland; also check the web for any updates.")




> Entering new AgentExecutor chain...

Invoking: `Simple_QA` with `{'tool_input': 'What is the capital of the Netherlands?'}`


The capital of the Netherlands is Amsterdam.
Invoking: `Web_Search` with `{'tool_input': 'current capital of the Netherlands'}`


[{'title': 'Capital of the Netherlands - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Capital_of_the_Netherlands', 'content': 'Wikipedia\nThe Free Encyclopedia\n\n## Contents\n\n# Capital of the Netherlands\n\nIn the Netherlands, the Constitution refers to Amsterdam as the capital city. However, since 1588, the States General and the Executive Branch, along with the Supreme Court and the Council of State "Council of State (Netherlands)"), have been situated in The Hague as the seat of government. Since the 1983 revision of the Constitution of the Netherlands, Article 32 mentions that "the King shall be sworn in and inaugurated as soon as possible in the capital city, Amsterdam". It is the only reference in the document statin

In [38]:
#7) Self-Ask with Search (clarifies, then searches)

search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=TavilySearchResults(max_results=1).run,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")




> Entering new AgentExecutor chain...
Yes.  
Follow up: When was the first release of LangChain?  
Intermediate answer: [{'title': 'LangChain - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/LangChain', 'content': '| LangChain |\n\n| Image 4: 🦜️🔗, the parrot and chain emojis |\n| Developer | Harrison Chase |\n| Initial release | October 2022 |\n|  |\n| Stable release | 0.1.16( / 11 April 2024; 2 years ago(11 April 2024) |\n|  |\n| Written in | Python "Python (programming language)") and JavaScript |\n| Type | Software framework for large language model application development |\n| License | MIT License |\n| Website | LangChain.com |\n| Repository "Repository (version control)") | github.com/langchain-ai/langchain |\n\n   Image 5Free and open-source software portal [...] ## History\n\n[edit]\n\nLangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated a